# Exploring the OpenAI API: Tokens, Costs, and Usage

This notebook demonstrates how to interact with the **OpenAI API** from Python in a reproducible classroom or research environment.  
The focus is on understanding how **tokenization**, **model usage**, and **costs per token** work in practice.





##  What the Notebook Does


1. **Connects to the OpenAI API** using the shared API key.  
2. **Sends example prompts** to small and large models (e.g., `gpt-4-turbo` or `gpt-4o-mini`) to illustrate response quality and cost trade-offs.  
3. **Explores tokenization** — how text is converted into tokens and how token counts vary by model.  
4. **Calculates API usage costs**, showing how prompt length and model choice affect pricing.  
5. **Visualizes results**, helping students understand the relationship between:
   - Input text length (number of tokens)
   - Model type and context window
   - Cost per request

##  Learning Goals

- Understand what a **token** is and how it differs from characters or words.  
- Learn to estimate and monitor **API usage costs**.  
- Gain experience working with **environment variables** and best practices for secret management.  
- Build intuition for **trade-offs between model size, latency, and price** in practical applications.

In [ ]:
import os
from IPython.display import display
import ipywidgets as widgets
import pandas as pd
import matplotlib.pyplot as plt
import requests
import json

In [ ]:
try:
    from dotenv import load_dotenv
except:
    !pip install python-dotenv
    from dotenv import load_dotenv

In [ ]:
try:
    from openai import OpenAI
except ImportError:
    !pip install openai
    from openai import OpenAI


##  API Key Setup

To keep credentials secure, the API key is **not stored directly in this notebook**.  

*The API key is linked  to my credit card, so if it gets out the charges could add up.  If I put the API key on Github it will be automatically flagged.* 

Instead, it is stored in a `.env` file inside a shared directory (`../shared/.env`) with a line like: `openai_API_KEY="..."`

In [ ]:
# FA 25 Directory for Data 88E - Instructor
%ls -a /home/jovyan/_shared/bcourses-1547593-readwrite/

In [ ]:
# FA 25 Directory for Data 88E - Student 
%ls -a /home/jovyan/_shared/bcourses-1547593-readonly/

In [ ]:

ENV_PATH = "/home/jovyan/_shared/bcourses-1547593-readonly/.env" # student path
#ENV_PATH = "/home/jovyan/_shared/bcourses-1547593-readwrite/.env"   # instructor path

loaded = load_dotenv(ENV_PATH)
print("dotenv loaded:", loaded)

openai_api_key = os.getenv("openai_API_KEY")

print("OpenAI key loaded:", "✅" if openai_api_key else "❌ not found")

( option to manually load ) 

In [ ]:
# this cell is just if you want to manually load a different API Key 
#openai_API_KEY ="  "

### Notes for Instructors

- The shared `.env` file allows multiple users on the same DataHub instance or Jupyter environment to access a single institutional API key without embedding secrets in their notebooks.  
- Students should **never print the API key** or share the `.env` file contents publicly.  
- The key can be rotated by updating the shared `.env` file; all dependent notebooks will continue to function.


##  The OpenAI Python Package

The **OpenAI Python package** provides a simple interface for interacting with OpenAI’s models—such as GPT, Whisper, and DALL·E—directly from Python code. It supports both synchronous and asynchronous API calls, making it easy to send prompts, generate completions, and analyze responses. The package handles authentication via an environment variable (`openai_API_KEY`) and returns structured results that can be easily integrated into data workflows, Jupyter notebooks, or applications for natural language processing, code generation, or AI-assisted analysis.

### Initializing the OpenAI Client

Once the API key is loaded from the environment, we create a client object that serves as our connection to the OpenAI API.  
This client will handle authentication and allow us to make requests to different models.  

We’ll initialize it like this:


In [ ]:
client = OpenAI(api_key = openai_api_key)

### Checking Available Models

Before making any API calls, it’s useful to list the models that your API key can access.  
The `client.models.list()` method returns all available model identifiers for your OpenAI account, such as `gpt-4o`, `gpt-4-turbo`, and smaller variants like `gpt-4o-mini`.  
Listing these helps confirm the correct model names to use in later API requests.

In [ ]:
models = client.models.list()
print([m.id for m in models])

In [ ]:
# Send a chat message to GPT-3.5
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": "You are a UC Berkeley Economics Student"},
        {"role": "user", "content": "Explain who pays the burden of tariffs"}
    ]
)

# Display response
print(response.choices[0].message.content)

## Basic Chat Completion Example

To demonstrate the simplest API call, we can send a chat-style request to one of the OpenAI language models.  
Here, we use `client.chat.completions.create()` to send a short conversation.  
The model responds based on the system and user messages provided.

In this example, the system message defines the context (“You are a UC Berkeley Economics student”), and the user asks a question (“Explain who pays the burden of tariffs”).  
The model returns a text completion that we can extract and display from the `response` object.

This basic pattern—system message, user message, and model reply—is the foundation of all chat-based interactions with OpenAI models.

## Token Pricing 
https://platform.openai.com/docs/pricing#text-tokens


## OpenAI Token Pricing (as of December 2025)

Prices below are converted from OpenAI’s official rates (per 1M tokens → per 1K tokens).  
Source: https://platform.openai.com/docs/pricing#text-tokens

| **Model**           | **Input (per 1K)** | **Cached Input (per 1K)** | **Output (per 1K)** |
|---------------------|---------------------|-----------------------------|-----------------------|
| **gpt-5.1**         | \$0.00125           | \$0.000125                  | \$0.01000             |
| **gpt-5**           | \$0.00125           | \$0.000125                  | \$0.01000             |
| **gpt-5-mini**      | \$0.00025           | \$0.000025                  | \$0.00200             |
| **gpt-5-nano**      | \$0.00005           | \$0.000005                  | \$0.00040             |
| **gpt-5.1-chat-latest** | \$0.00125       | \$0.000125                  | \$0.01000             |
| **gpt-5-chat-latest**  | \$0.00125         | \$0.000125                  | \$0.01000             |
| **gpt-5.1-codex**   | \$0.00125           | \$0.000125                  | \$0.01000             |
| **gpt-5-codex**     | \$0.00125           | \$0.000125                  | \$0.01000             |
| **gpt-5-pro**       | \$0.01500           | —                           | \$0.12000             |
| **gpt-4.1**         | \$0.00200           | \$0.00050                   | \$0.00800             |
| **gpt-4.1-mini**    | \$0.00040           | \$0.00010                   | \$0.00160             |
| **gpt-4.1-nano**    | \$0.00010           | \$0.000025                  | \$0.00040             |
| **gpt-4o**          | \$0.00250           | \$0.00125                   | \$0.01000             |
| **gpt-4o-2024-05-13** | \$0.00500         | —                           | \$0.01500             |
| **gpt-4o-mini**     | \$0.00015           | \$0.000075                  | \$0.00060             |

In [ ]:
# First a csv 
csv_text = """Model,Input_per_1K,CachedInput_per_1K,Output_per_1K
gpt-5.1,0.00125,0.000125,0.01000
gpt-5,0.00125,0.000125,0.01000
gpt-5-mini,0.00025,0.000025,0.00200
gpt-5-nano,0.00005,0.000005,0.00040
gpt-5.1-chat-latest,0.00125,0.000125,0.01000
gpt-5-chat-latest,0.00125,0.000125,0.01000
gpt-5.1-codex,0.00125,0.000125,0.01000
gpt-5-codex,0.00125,0.000125,0.01000
gpt-5-pro,0.01500,,0.12000
gpt-4.1,0.00200,0.00050,0.00800
gpt-4.1-mini,0.00040,0.00010,0.00160
gpt-4.1-nano,0.00010,0.000025,0.00040
gpt-4o,0.00250,0.00125,0.01000
gpt-4o-2024-05-13,0.00500,,0.01500
gpt-4o-mini,0.00015,0.000075,0.00060
"""

with open("openai_pricing_dec2025.csv", "w") as f:
    f.write(csv_text)

In [ ]:
# Load the CSV
df = pd.read_csv("openai_pricing_dec2025.csv")

# Optionally sort by output cost (most expensive → cheapest)
df = df.sort_values("Output_per_1K", ascending=False)

# Make a bar chart of output cost per 1K tokens
plt.figure(figsize=(10, 5))
plt.bar(df["Model"], df["Output_per_1K"])

plt.title("OpenAI Model Output Cost per 1K Tokens (Dec 2025)")
plt.xlabel("Model")
plt.ylabel("USD per 1K output tokens")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Token prices (per 1K tokens) — December 2025 OpenAI pricing
# Python Dictionary 
token_prices = {
    "gpt-5":     {"input": 0.00125, "output": 0.01000},
    "gpt-5.1":   {"input": 0.00125, "output": 0.01000},
    "gpt-5-mini":{"input": 0.00025, "output": 0.00200},
    "gpt-5-nano":{"input": 0.00005, "output": 0.00040},
}

##  A widget to calculate costs of token consumption ?
### Interactive Token Cost Estimator

Use the controls below to select a model and enter the number of input and output tokens.  
The widget will calculate the estimated cost of a single API call using the December 2025 OpenAI pricing.

In [ ]:
# Widgets
model_selector = widgets.Dropdown(
    options=list(token_prices.keys()),value="gpt-5",description='Model:',)

input_tokens = widgets.IntText(value=1000,description='Input Tokens:',)

output_tokens = widgets.IntText(value=500,description='Output Tokens:',)

estimate_button = widgets.Button(description="Estimate Cost",button_style="success")

cost_display = widgets.Label(value="")

# Define the estimator
def estimate_cost(b):
    model = model_selector.value
    input_count = input_tokens.value
    output_count = output_tokens.value
    prices = token_prices[model] 
    cost = (input_count / 1000) * prices["input"] + (output_count / 1000) * prices["output"]
    cost_display.value = f"💲 Estimated Cost: ${cost:.6f}"

estimate_button.on_click(estimate_cost)

# Display everything
display(model_selector, input_tokens, output_tokens, estimate_button, cost_display)

### Example: Asking the Model an Economics Question

In this example, we send a question to the GPT-5-nano model using a simple two-message chat setup:

- A **system message** that sets the model’s persona (“UC Berkeley Economics student”)
- A **user message** asking: *“Who pays the burden of tariffs?”*

The model returns an answer along with token usage, allowing us to inspect both the explanation and its computational cost.

### What the "Display response" code does

After sending the prompt to the model, the API returns a `response` object that includes:

- `response.choices[0].message.content` — the model’s text output  
- `response.usage` — the token usage for the request  

The first print statement displays the model’s answer.  
The second block shows how many tokens were used for the prompt, completion, and total.  
This helps us understand both the quality of the response **and** the computational cost.

In [ ]:
# Send a chat message to GPT-5-nano
response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "system", "content": "You are a UC Berkeley Economics Student"},
        {"role": "user", "content": "Explain who pays the burden of tariffs"}
    ]
)

# Display response
print(response.choices[0].message.content)

# Display token usage
print("\n🔢 Token Usage:")
print(f"Prompt tokens: {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")


### Estimating the Cost of an API Call

Using the token prices for each model (cost per 1K input and output tokens), we can estimate how much this specific API call costs.  
We multiply the number of prompt tokens by the input price, and the number of completion tokens by the output price, then add them together.  
This lets us see, in dollars, how expensive a single model call is.

### How the Cost Calculation Works

OpenAI charges separately for **input tokens** (the prompt you send)  
and **output tokens** (the model’s response). Prices are given **per 1,000 tokens**.

To estimate the cost of a single API call, we use:

$$
\text{Cost} 
= \left(\frac{\text{input tokens}}{1000}\right) \cdot \text{input price per 1K}
\;+\;
  \left(\frac{\text{output tokens}}{1000}\right) \cdot \text{output price per 1K}
$$

This lets us translate token usage into a dollar amount:

- More **prompt tokens** → higher input cost  
- More **response tokens** → higher output cost  
- Larger models (e.g., GPT-5 vs. GPT-5-nano) have different prices per 1K tokens

The cell below applies this formula using the model’s token usage from the API response.

In [ ]:
# ---- Estimate API call cost for this response ----

model_name = "gpt-5-nano"  # keep in sync with the model you used above

# Get prices (per 1K tokens)
input_price_per_1k = token_prices[model_name]["input"]
output_price_per_1k = token_prices[model_name]["output"]

prompt_tokens = response.usage.prompt_tokens
completion_tokens = response.usage.completion_tokens
total_tokens = response.usage.total_tokens

# Cost = (tokens / 1000) * price_per_1k
prompt_cost = (prompt_tokens / 1000) * input_price_per_1k
completion_cost = (completion_tokens / 1000) * output_price_per_1k
total_cost = prompt_cost + completion_cost

print("\n💰 Cost Estimate:")
print(f"Model: {model_name}")
print(f"Prompt tokens: {prompt_tokens} → cost = ${prompt_cost:.6f}")
print(f"Completion tokens: {completion_tokens} → cost = ${completion_cost:.6f}")
print(f"Total tokens: {total_tokens} → total cost = ${total_cost:.6f}")

## Use this cell to do a new Query 

In [ ]:
response = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "You are a UC Berkeley Economics Student"},
        {"role": "user", "content": "What are some awesome upper div Econ Classes at Cal?"}
    ],
    temperature=0.7,               # creativity level (0 = deterministic, 1 = max randomness)
    top_p=1.0,                     # nucleus sampling (used instead of temperature, but can be combined)
    presence_penalty=0.5,         # encourages new topics
    frequency_penalty=0.3,        # discourages repetition
    max_tokens=200,               # max length of the response
    stop=None                     # can be a list of strings to stop generation early (e.g., ["\n", "END"])
)

# Display the response text
print("📘 Response:")
print(response.choices[0].message.content)

# Display token usage
print("\n🔢 Token Usage:")
print(f"Prompt tokens: {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")

## Longer Input Text 
What if we get the model to read the Textbook Chapter on Central Banks

https://data88e.org/textbook/content/09-macro/CentralBanks.html

### Turning a Textbook Chapter into Model Input

Now let’s see what happens when we send **a lot** of text to the model.  
In this example, we load an entire chapter of the Data 88E textbook directly from GitHub.

The code below:

1. Downloads the `CentralBanks.ipynb` notebook using its raw GitHub URL.  
2. Reads the notebook JSON and extracts all **markdown cells** (the explanatory text, not the code).  
3. Joins those markdown cells into one long string called `chapter_text`.  
4. Prints the number of characters in `chapter_text` and shows the first 800 characters as a preview.

We’ll then use `chapter_text` as a long prompt for the model and see how many tokens it uses, and how that affects the cost.





In [ ]:
# Raw GitHub URL for the notebook
url = "https://raw.githubusercontent.com/data-88e/textbook/master/content/09-macro/CentralBanks.ipynb"

resp = requests.get(url)
resp.raise_for_status()

nb = resp.json()

# Extract markdown cells and join them into one big string
markdown_cells = []
for cell in nb["cells"]:
    if cell.get("cell_type") == "markdown":
        # each cell["source"] is a list of lines
        markdown_cells.append("".join(cell.get("source", [])))

chapter_text = "\n\n".join(markdown_cells)

print("Characters in chapter_text:", len(chapter_text))
print(chapter_text[:800])  # peek at the first ~800 chars

In [ ]:
system_prompt = "You are a UC Berkeley economics student explaining macroeconomic concepts to your peers."

user_prompt = (
    "Here is a long chapter from the Data 88E textbook about central banks. "
    "Please give a concise, one-paragraph synthesis aimed at a first-year economics student.\n\n"
    + chapter_text
)

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
)

print(response.choices[0].message.content)

print("\n🔢 Token usage:")
print(f"Prompt tokens:     {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")
print(f"Total tokens:      {response.usage.total_tokens}")
print("\n💰 Cost Estimate:")
print(f"Model: {model_name}")
print(f"Prompt tokens: {prompt_tokens} → cost = ${prompt_cost:.6f}")
print(f"Completion tokens: {completion_tokens} → cost = ${completion_cost:.6f}")
print(f"Total tokens: {total_tokens} → total cost = ${total_cost:.6f}")

## One More Angle - How about using the commands to be a Data 88E tutor

Let's go  over and find the system prompt for the Data 88E tutor

https://github.com/data-88e/88e_training_material/blob/main/Data88E_Tutor_System_Prompt_v2.md

One thing we can do is just read the system prompt into memory 


In [ ]:
url = "https://raw.githubusercontent.com/data-88e/88e_training_material/main/Data88E_Tutor_System_Prompt_v2.md"

resp = requests.get(url)
resp.raise_for_status()  # fail loudly if GitHub can't be reached

data88e_system_prompt = resp.text

print("Loaded Data 88E Tutor System Prompt (chars):", len(data88e_system_prompt))
print(data88e_system_prompt[:600])  # peek at start

## Feed in system prompt as `system`

In [ ]:
question = "Who bears the burden of tariffs, consumers or producers?"

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "system", "content": data88e_system_prompt},
        {"role": "user", "content": question},
    ],
)

# Show the tutor-style answer
print(response.choices[0].message.content)

# Show token usage
usage = response.usage
print("\n🔢 Token usage:")
print(f"Prompt tokens:     {usage.prompt_tokens}")
print(f"Completion tokens: {usage.completion_tokens}")
print(f"Total tokens:      {usage.total_tokens}")

## Looking at token counts using OpenAI utility 

In [ ]:
try:
    import tiktoken
except ModuleNotFoundError:
    !pip install tiktoken --quiet
    import tiktoken

In [ ]:
encoding = tiktoken.encoding_for_model("gpt-5-nano")

system_tokens = len(encoding.encode(data88e_system_prompt))
print("System prompt tokens:", system_tokens)

In [ ]:
# Count tokens in the chapter text
chapter_tokens = len(encoding.encode(chapter_text))
print("Token count for chapter_text:", chapter_tokens)

In [ ]:
# Count tokens of the output text ?

# The model's output text
assistant_text = response.choices[0].message.content

# Count tokens
assistant_tokens = len(encoding.encode(assistant_text))



In [ ]:
# Text generated by the model
assistant_text = response.choices[0].message.content

# Encode the text into token IDs
tokens = encoding.encode(assistant_text)

print("🔢 Number of tokens in assistant output:", len(tokens))
print("\n🔤 Token IDs:")
print(tokens)

# Decode tokens one by one (human-readable pieces)
decoded_tokens = [encoding.decode([t]) for t in tokens]

print("\n🧩 Decoded tokens (pieces of text):")
print(decoded_tokens)

⭐ Final Message

If you got this far, you’re going far.
Because once you understand how a word becomes a token,
you suddenly see the whole matrix —
and realize we’re just visitors in the robots’ world now. 😄

Keep exploring, keep measuring, keep asking good questions.
The models get bigger, the tokens get cheaper,
but your intuition is the real superpower.

Let’s go. 🚀



![Data 88E Image](https://data88e.org/assets/images/blue_text.png)